# Thí nghiệm Big Data: Multimodal Fake News Detection (Fakeddit Subset)
Notebook này được thiết kế tự động để chạy trên Google Colab nhằm chứng minh năng lực của mô hình Cross-Attention trên tập dữ liệu lớn (Big Data).

## Bước 1: Khởi tạo môi trường và Clone mã nguồn
Tự động xóa thư mục cũ (nếu có) trước khi clone để tránh lỗi khi chạy lại.

In [1]:
# Xóa thư mục cũ nếu đã tồn tại để tránh lỗi git clone
!rm -rf multimodal-fake-news-detection
print("Đang clone mã nguồn từ GitHub...")
!git clone https://github.com/btsuu25-dev/multimodal-fake-news-detection.git
%cd multimodal-fake-news-detection
print("Đang cài đặt thư viện (mất khoảng 15-20 phút lần đầu)...")
!pip install -r requirements.txt -q
!pip install datasets tqdm requests pandas -q
print("✅ Bước 1 hoàn tất!")

Đang clone mã nguồn từ GitHub...
Cloning into 'multimodal-fake-news-detection'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 117 (delta 40), reused 100 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 527.25 KiB | 9.25 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/multimodal-fake-news-detection
Đang cài đặt thư viện (mất khoảng 15-20 phút lần đầu)...
✅ Bước 1 hoàn tất!


## Bước 1.5: Tối ưu hóa hiệu suất (Ép xung GPU)
Tự động tăng `BATCH_SIZE` lên 64 và `NUM_WORKERS` lên 4 để tận dụng tối đa GPU 16GB VRAM của Colab T4.

> **Lưu ý:** Dùng Regex để thay đúng tên biến hằng số `BATCH_SIZE` và `NUM_WORKERS` (viết hoa) trong file nguồn.

In [2]:
import os, re

files_to_edit = ['src/train.py', 'src/train_baseline.py']

for file_path in files_to_edit:
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # Dùng Regex thay đúng tên biến IN HOA (BATCH_SIZE, NUM_WORKERS)
        content = re.sub(r'BATCH_SIZE\s*=\s*\d+', 'BATCH_SIZE  = 64', content)
        content = re.sub(r'NUM_WORKERS\s*=\s*\d+', 'NUM_WORKERS  = 4', content)

        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f'✅ Đã ép xung: {file_path}')
    else:
        print(f'❌ Không tìm thấy file: {file_path}')

print('\n🚀 Hoàn tất! BATCH_SIZE=64 và NUM_WORKERS=4 đã được áp dụng cho cả 2 file train!')

✅ Đã ép xung: src/train.py
✅ Đã ép xung: src/train_baseline.py

🚀 Hoàn tất! BATCH_SIZE=64 và NUM_WORKERS=4 đã được áp dụng cho cả 2 file train!


## Bước 2: Tải và Tiền xử lý dữ liệu Fakeddit
Dùng đa luồng tải ảnh từ 80.000 link để lọc ra chính xác 30.000 ảnh sống sót.

In [3]:
import os
import requests
import pandas as pd
from datasets import load_dataset
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

print("Đang tải dữ liệu gốc Fakeddit từ máy chủ Parquet tốc độ cao...")

dataset = load_dataset('AdoCleanCode/Fakeddit', split='train', streaming=False)
df = dataset.to_pandas()

df = df[df['hasImage'] == True]
df['2_way_label'] = df['label_name'].apply(lambda x: 0 if x == 'True' else 1)

# Lấy 80.000 link để bù trừ cho tỷ lệ link chết ~50%
df = df[['image_url', 'text', '2_way_label']].dropna().head(80000)

os.makedirs('data/images', exist_ok=True)
os.makedirs('data/splits', exist_ok=True)

valid_rows = []
MAX_SAMPLES = 30000  # Mục tiêu chốt cứng 30.000 mẫu

def download_image(row):
    if len(valid_rows) >= MAX_SAMPLES:
        return

    url = row['image_url']
    img_name = str(url).split('/')[-1].split('?')[0]
    img_path = os.path.join('data/images', img_name)

    if not os.path.exists(img_path):
        try:
            res = requests.get(url, timeout=3)
            if res.status_code == 200:
                with open(img_path, 'wb') as f:
                    f.write(res.content)
                if len(valid_rows) < MAX_SAMPLES:
                    valid_rows.append({
                        'image_path': img_path,
                        'text': row['text'],
                        'label': row['2_way_label']
                    })
        except:
            pass

print(f"Bắt đầu quét qua 80.000 link để chắt lọc đúng {MAX_SAMPLES} ảnh sống...")
rows_list = [row for _, row in df.iterrows()]
with ThreadPoolExecutor(max_workers=32) as executor:
    list(tqdm(executor.map(download_image, rows_list), total=len(rows_list)))

valid_rows = valid_rows[:MAX_SAMPLES]
print(f"\nĐã tải và chắt lọc thành công CHÍNH XÁC {len(valid_rows)} mẫu dữ liệu Fakeddit gốc hợp lệ!")

Đang tải dữ liệu gốc Fakeddit từ máy chủ Parquet tốc độ cao...


README.md:   0%|          | 0.00/696 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 73.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/794660 [00:00<?, ? examples/s]

Bắt đầu quét qua 80.000 link để chắt lọc đúng 30000 ảnh sống...


100%|██████████| 80000/80000 [04:31<00:00, 294.90it/s]


Đã tải và chắt lọc thành công CHÍNH XÁC 23512 mẫu dữ liệu Fakeddit gốc hợp lệ!


## Bước 2.5: Lọc ảnh bị hỏng (Corrupt Filter)
Kiểm tra và loại bỏ các ảnh bị hỏng file trước khi lưu vào CSV để đảm bảo 100% dữ liệu train là ảnh thật.

In [4]:
from PIL import Image

print("Đang kiểm tra tính toàn vẹn của ảnh...")

def is_valid_image(path):
    try:
        img = Image.open(path)
        img.verify()  # Kiểm tra file ảnh không bị hỏng
        return True
    except:
        return False

before = len(valid_rows)
valid_rows = [row for row in valid_rows if is_valid_image(row['image_path'])]
after = len(valid_rows)

print(f"Giữ lại {after}/{before} ảnh hợp lệ (đã lọc bỏ {before - after} ảnh bị hỏng)")
print("✅ Dữ liệu đã được làm sạch 100%!")

Đang kiểm tra tính toàn vẹn của ảnh...
Giữ lại 23480/23512 ảnh hợp lệ (đã lọc bỏ 32 ảnh bị hỏng)
✅ Dữ liệu đã được làm sạch 100%!


## Bước 3: Chia tập dữ liệu (Train/Val/Test)

In [5]:
from sklearn.model_selection import train_test_split
import pandas as pd

final_df = pd.DataFrame(valid_rows)

# Chia theo tỷ lệ 80-10-10
train_df, temp_df = train_test_split(final_df, test_size=0.2, random_state=42, stratify=final_df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_df.to_csv('data/splits/train.csv', index=False)
val_df.to_csv('data/splits/val.csv', index=False)
test_df.to_csv('data/splits/test.csv', index=False)

print("Đã lưu các file phân chia dữ liệu vào thư mục data/splits/")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Đã lưu các file phân chia dữ liệu vào thư mục data/splits/
Train: 18784, Val: 2348, Test: 2348


## Bước 4: Huấn luyện Mô hình Concat (Baseline)
Cùng xem sức mạnh của kiến trúc đơn giản khi gặp Big Data.

In [6]:
import os
os.environ['PYTHONPATH'] = '.'
!python src/train_baseline.py

--- NẠP DỮ LIỆU & ENCODER (M1, M2, M3) ---
[INFO] Đang tải mô hình CLIP Image Encoder: openai/clip-vit-base-patch32...
[INFO] HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"
[INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"
config.json: 100% 4.19k/4.19k [00:00<00:00, 12.8MB/s]
[INFO] HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
[INFO] HTTP Request: HEAD https://huggingface.co/openai/cli

## Bước 5: Huấn luyện Mô hình Cross-Attention (Mô hình chính)
Kỳ vọng: Cross-Attention sẽ phát huy sức mạnh bắt chéo đặc trưng trên tập dữ liệu đa dạng và lớn hơn này.

In [7]:
import os
os.environ['PYTHONPATH'] = '.'
!python src/train.py

[INFO] Device: cuda | Mixed Precision (AMP): True
[INFO] Hyperparameters: EPOCHS=15, BATCH=64, LR=0.0002
[INFO] Đang nạp dữ liệu (M1 – FakeNewsDataset)...
[INFO] Đã tải 18784 mẫu từ: data/splits/train.csv
[INFO] Đã tải 2348 mẫu từ: data/splits/val.csv
[INFO]   Train: 18784 mẫu
[INFO]   Val  : 2348 mẫu
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[INFO] Đang khởi tạo CrossModalFND (M5)...
[INFO] Đang tải mô hình CLIP Image Encoder: openai/clip-vit-base-patch32...
[INFO] HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.j

## Bước 6: Đánh giá và Xuất báo cáo (HTML)

In [8]:
!python src/evaluate.py
!python src/evaluation/html_report.py
print("Hoàn tất! Bạn có thể tải file results/dashboard.html về máy tính để xem kết quả so sánh cuối cùng!")

[INFO] Device: cuda

=== NAP DU LIEU TEST ===
[INFO] Đã tải 2348 mẫu từ: data/splits/test.csv
  Test set: 2348 mau

  [A] MODEL-LEVEL EVALUATION
  Danh gia tung mo hinh rieng le tren test.csv
  Dang khoi tao CLIP Encoders (dung chung)...
[INFO] Đang tải mô hình CLIP Image Encoder: openai/clip-vit-base-patch32...
[INFO] HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"
[INFO] HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-

In [9]:
# Nén toàn bộ kết quả thành 1 file zip để tải về dễ dàng
import shutil
shutil.make_archive('fakeddit_results', 'zip', 'results')
print("✅ File fakeddit_results.zip đã sẵn sàng để tải!")
# Sau đó click chuột phải vào file fakeddit_results.zip bên panel Tệp → Download


✅ File fakeddit_results.zip đã sẵn sàng để tải!
